<a href="https://colab.research.google.com/github/contreras-juan/procesamiento_lenguaje_natural/blob/main/Proyectos/02_Proyecto_Modelos_Probabilisticos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="color: #FECB05; text-align: center;">Proyecto 2: modelos probabilísticos de PLN</h1>

<h2 style="color: #007ACC;">Autores</h2>

- [Juan Felipe Contreras Alcívar](https://www.linkedin.com/in/juanf-contreras/)

---

<h2 style="color: #007ACC;">Instrucciones</h2>

**Objetivo:** construir y evaluar **modelos probabilísticos de PLN** para **predecir** y **clasificar** texto en un dominio real, aplicando lo aprendido en el módulo 2 (N-gramas, suavizado, perplejidad, Naive Bayes, máxima entropía, HMM, Viterbi y MEMM).

**Modalidad:** grupos de 2 a 3 personas.

El proyecto tiene dos partes sobre **el mismo corpus**, que debe estar preferiblemente **en español** y pertenecer a un dominio concreto. Algunas ideas: noticias de una sección, sentencias judiciales, historias clínicas anonimizadas, letras de canciones de varios géneros, discursos de distintos políticos, comentarios de clientes con categorías, mensajes de soporte técnico.

**Parte A — Predicción: un modelo de lenguaje para el dominio**

1. Un modelo de N-gramas con `<UNK>` y al menos dos técnicas de suavizado (por ejemplo add-k e interpolación), con hiperparámetros elegidos en **validación** y perplejidad reportada en **prueba**.
2. Una comparación de la perplejidad para distintos valores de $N$ y distintos tamaños del corpus de entrenamiento (curva de aprendizaje).
3. Una demostración de **autocompletado** y de **generación** con al menos dos estrategias de muestreo.

**Parte B — Clasificación** (elige **una** opción)

- **B1. Clasificación de documentos:** al menos tres clases (tema, autor, género, departamento...). Compara Naive Bayes, máxima entropía / regresión logística y el clasificador generativo basado en modelos de lenguaje por clase (taller 03), con características de N-gramas.
- **B2. Etiquetado de secuencias:** anota (o consigue) un corpus con etiquetas por palabra para una tarea del dominio (entidades específicas como medicamentos, normas o productos; o etiquetado gramatical). Compara una línea base, un HMM con Viterbi y un MEMM. Evalúa por entidades si es NER.

En ambos casos: validación sin fuga de información, métricas adecuadas, análisis de errores cuantificado y reflexión ética.

**Entregables:**

- Este cuaderno completo, ejecutado de principio a fin sin errores (*Kernel → Restart & Run All*).
- Una presentación de 10 minutos con los hallazgos principales.

Puedes añadir las celdas que necesites en cada sección. Justifica cada decisión técnica.

<h2 style="color: #007ACC;">Tabla de contenido</h2>

- [<span style="color: #005C99;">Definición del problema y del corpus</span>](#definicion-del-problema-y-del-corpus)
- [<span style="color: #005C99;">Preparación de los datos</span>](#preparacion-de-los-datos)
- [<span style="color: #005C99;">Parte A: modelo de lenguaje</span>](#parte-a-modelo-de-lenguaje)
- [<span style="color: #005C99;">Parte A: autocompletado y generación</span>](#parte-a-autocompletado-y-generacion)
- [<span style="color: #005C99;">Parte B: modelos de clasificación</span>](#parte-b-modelos-de-clasificacion)
- [<span style="color: #005C99;">Parte B: evaluación y análisis de errores</span>](#parte-b-evaluacion-y-analisis-de-errores)
- [<span style="color: #005C99;">Conclusiones e implicaciones éticas</span>](#conclusiones-e-implicaciones-eticas)
- [<span style="color: #005C99;">Rúbrica de evaluación</span>](#rubrica-de-evaluacion)

---

<h2 style="color: #007ACC;">Instalación de librerías</h2>

Ejecuta esta celda si trabajas en Google Colab o en un entorno nuevo.

In [ ]:
# Instalación de librerías necesarias para esta sesión
# (útil en Google Colab o en un entorno nuevo)
%pip install -q numpy pandas matplotlib seaborn scikit-learn nltk

In [ ]:
import math
import random
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, f1_score, ConfusionMatrixDisplay

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

<a id="definicion-del-problema-y-del-corpus"></a>
<h2 style="color: #007ACC;">Definición del problema y del corpus</h2>

- ¿Qué dominio elegiste y por qué? ¿Quién usaría un modelo de lenguaje o un clasificador para este dominio y para qué?
- Describe la fuente de los datos, su licencia, su fecha y su tamaño (oraciones, tokens, documentos por clase).
- Para la Parte B: ¿qué opción elegiste (B1 o B2)? ¿Cómo se obtuvieron las etiquetas? Si etiquetaste tú, describe el criterio y mide el acuerdo entre anotadores en una muestra.

In [ ]:
# Escribe tu código aquí

<a id="preparacion-de-los-datos"></a>
<h2 style="color: #007ACC;">Preparación de los datos</h2>

- Segmenta en oraciones y tokeniza (cuaderno 01). Justifica las decisiones (minúsculas, números, puntuación, expresiones de varias palabras).
- Divide en entrenamiento, validación y prueba. Si los documentos tienen autor, fecha o fuente, piensa si la división debe respetarlos para evitar que el modelo "haga trampa" (por ejemplo, entrenar y probar con capítulos del mismo libro).
- Analiza la dispersión: N-gramas distintos y porcentaje de singletons para $N = 1, \dots, 4$; tasa de palabras fuera del vocabulario en validación.

In [ ]:
# Escribe tu código aquí

<a id="parte-a-modelo-de-lenguaje"></a>
<h2 style="color: #007ACC;">Parte A: modelo de lenguaje</h2>

- Implementa (o adapta del taller 03) los modelos de N-gramas con `<UNK>` y suavizado.
- Elige los hiperparámetros (k, λ) en **validación** y reporta la perplejidad en **prueba** en una tabla.
- Grafica la perplejidad en validación contra el tamaño del corpus de entrenamiento (por ejemplo, 10 %, 25 %, 50 %, 100 %) para unigramas, bigramas y trigramas. ¿Cuántos datos harían falta para que el trigrama valga la pena?
- Opcional: compara con `nltk.lm.KneserNeyInterpolated`.

In [ ]:
# Escribe tu código aquí

<a id="parte-a-autocompletado-y-generacion"></a>
<h2 style="color: #007ACC;">Parte A: autocompletado y generación</h2>

- Implementa una función de autocompletado (siguientes $k$ palabras más probables) y muéstrala con prefijos típicos del dominio.
- Genera texto con al menos dos estrategias (voraz, *top-k*, temperatura) y comenta las diferencias.
- ¿El texto generado reproduce fragmentos literales del corpus? Mide qué porcentaje de los trigramas generados aparece tal cual en entrenamiento. ¿Qué riesgos tiene esto si el corpus contiene información sensible?

In [ ]:
# Escribe tu código aquí

<a id="parte-b-modelos-de-clasificacion"></a>
<h2 style="color: #007ACC;">Parte B: modelos de clasificación</h2>

- **B1:** entrena Naive Bayes, regresión logística / MaxEnt y el clasificador por modelos de lenguaje. Explora características de unigramas, bigramas y alguna característica diseñada a mano (al estilo del cuaderno 07).
- **B2:** entrena la línea base de etiqueta más frecuente, un HMM con tratamiento de palabras desconocidas y un MEMM con características ricas; decodifica con Viterbi.
- Selecciona hiperparámetros y características en **validación**.

In [ ]:
# Escribe tu código aquí

<a id="parte-b-evaluacion-y-analisis-de-errores"></a>
<h2 style="color: #007ACC;">Parte B: evaluación y análisis de errores</h2>

- Presenta una tabla comparativa en **prueba** con las métricas adecuadas (F1 macro para B1; F1 por entidades o exactitud por token para B2).
- Matriz de confusión del mejor modelo y características más influyentes.
- Análisis de errores: al menos 10 ejemplos, clasificados por tipo de error y cuantificados.

In [ ]:
# Escribe tu código aquí

<a id="conclusiones-e-implicaciones-eticas"></a>
<h2 style="color: #007ACC;">Conclusiones e implicaciones éticas</h2>

- Resume los hallazgos de ambas partes: ¿qué funcionó, qué no y por qué?
- ¿Qué limitaciones de los modelos probabilísticos clásicos observaste (contexto corto, dispersión, características manuales)? ¿Cómo esperarías que las resolvieran los modelos neuronales del módulo 3?
- **Ética:** reflexiona con ejemplos de tus datos sobre al menos tres de estos puntos:
  - memorización y reproducción de información personal por el modelo de lenguaje;
  - sesgos del corpus que el autocompletado podría amplificar (estereotipos de género, regionales...);
  - consecuencias de los errores del clasificador o etiquetador si se usara para tomar decisiones;
  - representatividad del corpus: ¿qué variedades del español o qué grupos quedan fuera?

In [ ]:
# Escribe tu código aquí

<a id="rubrica-de-evaluacion"></a>
<h2 style="color: #007ACC;">Rúbrica de evaluación</h2>

| Criterio | Peso | Se espera |
|----------|------|-----------|
| Problema, corpus y preparación | 15 % | Dominio relevante; división sin fugas; análisis de dispersión que informa las decisiones |
| Parte A: modelo de lenguaje | 25 % | Suavizado correcto, hiperparámetros en validación, curva de aprendizaje interpretada |
| Parte A: autocompletado y generación | 10 % | Estrategias de muestreo comparadas; análisis de memorización |
| Parte B: modelado | 20 % | Comparación justa de al menos tres enfoques; diseño de características justificado |
| Parte B: evaluación y errores | 15 % | Métricas adecuadas; análisis de errores cuantificado |
| Conclusiones y ética | 10 % | Reflexión concreta, basada en los datos y resultados propios |
| Calidad del cuaderno y presentación | 5 % | Código limpio y reproducible; narrativa clara |